# Round 2: Data Preprocessing & Split

**Goal**: Prepare data for Primary and Secondary emotion classification.
- Load Combined Data
- Text Cleaning
- Label Encoding (Primary & Secondary)
- Stratified Split (on Primary)

In [ ]:
import pandas as pd
import re
import json
from sklearn.model_selection import train_test_split
import os

print("Libraries imported.")

## 1. Load Data

In [ ]:
file_path = 'round2/Combined_Emotions.xlsx'
df = pd.read_excel(file_path)
print(f"Total Samples: {len(df)}")
display(df.head())

## 2. Text Cleaning

In [ ]:
def clean_poem(text):
    if not isinstance(text, str):
        return str(text)
    text = text.strip()
    text = re.sub(r'\n+', '\n', text)
    text = re.sub(r'\s+([,.;!?])', r'\1', text)
    return text

df['cleaned_poem'] = df['Poem'].apply(clean_poem)
print("Text cleaning complete.")

## 3. Label Encoding (Primary & Secondary)

In [ ]:
# Primary
primary_labels = sorted(df['Primary'].unique())
primary_map = {label: idx for idx, label in enumerate(primary_labels)}
df['primary_id'] = df['Primary'].map(primary_map)

# Secondary
secondary_labels = sorted(df['Secondary'].dropna().unique())
secondary_map = {label: idx for idx, label in enumerate(secondary_labels)}
df['secondary_id'] = df['Secondary'].map(secondary_map)

print(f"Primary Labels: {len(primary_map)}")
print(f"Secondary Labels: {len(secondary_map)}")

# Save Maps
maps = {
    'primary_map': primary_map,
    'secondary_map': secondary_map
}

os.makedirs('round2', exist_ok=True)
with open('round2/label_maps.json', 'w') as f:
    json.dump(maps, f, indent=4)
print("Label maps saved to round2/label_maps.json")

## 4. Stratified Split (by Primary)
We stratify by *Primary* because it's the main task.

In [ ]:
train_df, val_df = train_test_split(
    df, 
    test_size=0.2, 
    stratify=df['primary_id'], 
    random_state=42
)

print(f"Train: {len(train_df)}")
print(f"Val:   {len(val_df)}")

train_df.to_excel('round2/train.xlsx', index=False)
val_df.to_excel('round2/val.xlsx', index=False)
print("Train/Val sets saved to round2/")